# ModPlant-RTN — Shared App Kernel Notebook

This research-oriented notebook keeps `module_ops`, `_hc_data`, recipe and objective settings editable in code. RTN compilation, CP-SAT planning, persistent Connect/Disconnect lifecycle, energy/CO₂ accounting, validation, plan rows, and Gantt rendering all come from the same modules used by the ModPlant-RTN desktop App.

In [1]:
%reset -f

In [2]:
from pathlib import Path
import importlib
import sys
import pandas as pd

_candidate_roots = [Path.cwd(), Path.cwd() / "RTN", Path.cwd().parent / "RTN"]
PROJECT_ROOT = next(
    (path.resolve() for path in _candidate_roots
     if (path / "scripts" / "notebook_helpers.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run Jupyter from the Module/RTN project or set the working directory accordingly.")
for _path in (PROJECT_ROOT, PROJECT_ROOT / "scripts"):
    if str(_path) not in sys.path:
        sys.path.insert(0, str(_path))

import notebook_helpers as rtn_nb
rtn_nb = importlib.reload(rtn_nb)
from sample_data import (
    sample_equipment_bindings,
    sample_module_ops,
)
from modplant_rtn.models import plant_model_from_rtn_inputs
from modplant_rtn.service import RTNSettings, export_master_recipe, optimize_recipe_ir

build_recipe_ir = rtn_nb.build_recipe_ir
auto_enrich_recipe_spec = rtn_nb.auto_enrich_recipe_spec
save_general_recipe_xml_from_ir = rtn_nb.save_general_recipe_xml_from_ir
display_df = rtn_nb.display_df
display_recipe_graph = rtn_nb.display_recipe_graph
display_process_plan_graph = rtn_nb.display_process_plan_graph
display_connection_graph = rtn_nb.display_connection_graph
plan_dataframe = rtn_nb.plan_dataframe

## Module configuration — directly editable

The App's canonical four-Module demo is loaded once into ordinary Python dictionaries. Edit `module_ops`, `_hc_data`, maximum volumes, or initial resources below before solving. Capability tuples are:

`(operation, parameter, usage_cost_EUR_per_invocation, energy_rate_kWh_per_s, co2_rate_kg_per_s, fixed_duration_s)`

Only Connect and Disconnect use `fixed_duration_s`; all recipe-dependent operation durations are calculated from the recipe and flow parameters.

In [3]:
# ---- EDITABLE MODPLANT INPUTS ----
# These are normal lists/dicts: edit, append or replace records directly.
module_ops = sample_module_ops()

# (Module, number_of_inputs, number_of_outputs)
# Example: change HC30 from one to four output ports with ("HC30", 4, 4).
_hc_data = [
    ("HC10", 3, 3),
    ("HC20", 3, 3),
    ("HC30", 4, 4),
    ("HC40", 4, 4),
]

module_maximum_volume = {"HC10": [10.0], "HC20": [15.0], "HC30": [10.0], "HC40": [30.0]}
module_resources = {"HC10": ["A", 10.0], "HC20": ["B", 10.0], "HC30": ["C", 10.0]}
equipment_bindings = sample_equipment_bindings()

# Always derive physical ports after editing `_hc_data`.
module_interfaces = {
    hc: [("Input", f"{hc}_In{i}") for i in range(1, num_in + 1)]
        + [("Output", f"{hc}_Out{i}") for i in range(1, num_out + 1)]
    for hc, num_in, num_out in _hc_data
}

# Example direct capability override (uncomment and adapt):
# module_ops["HC10"][0] = ("Draining", 0.1, 3.0, 0.0611111111, 0.0244444444, 0.0)

_operation_rows = []
for wb, capabilities in module_ops.items():
    for operation, parameter, usage_cost, energy_rate, co2_rate, fixed_duration in capabilities:
        _operation_rows.append({
            "Module": wb,
            "Operation": operation,
            "Parameter": parameter,
            "Usage cost (€ / invocation)": usage_cost,
            "Energy rate (kWh/s)": energy_rate,
            "CO₂e rate (kg/s)": co2_rate,
            "Fixed duration (s)": fixed_duration if operation.casefold() in {"connect", "disconnect"} else "",
        })

display_df("Module Operation Capabilities", pd.DataFrame(_operation_rows))
display_df("Module Interfaces", pd.DataFrame(
    [{"Module": wb, "Type": kind, "Port": port}
     for wb, ports in module_interfaces.items() for kind, port in ports]))
display_df("Module Maximum Volumes", pd.DataFrame(
    [{"Module": wb, "Maximum Volume (L)": values[0]}
     for wb, values in module_maximum_volume.items()]))
display_df("Module Initial Resources", pd.DataFrame(
    [{"Module": wb, "Material": values[0], "Quantity (L)": values[1]}
     for wb, values in module_resources.items()]))


Module Operation Capabilities


,Module,Operation,Parameter,Usage cost (€ / invocation),Energy rate (kWh/s),CO₂e rate (kg/s),Fixed duration (s)
0,HC10,Draining,0.1,3.0,0.061111,0.024444,
1,HC10,Filling,0.1,0.0,0.020833,0.008333,
2,HC10,Settling,,1.0,0.003333,0.001333,
3,HC10,Stirring,100,3.0,0.055556,0.022222,
4,HC10,Stirring,200,3.0,0.111111,0.044444,
5,HC10,Connect,,2.0,0.013889,0.005556,3.0
6,HC10,Disconnect,,2.0,0.013889,0.005556,2.0
7,HC10,None,,0.0,0.002222,0.000889,
8,HC20,Draining,0.1,3.0,0.076389,0.030556,
9,HC20,Filling,0.1,0.0,0.026042,0.010417,



Module Interfaces


,Module,Type,Port
0,HC10,Input,HC10_In1
1,HC10,Input,HC10_In2
2,HC10,Input,HC10_In3
3,HC10,Output,HC10_Out1
4,HC10,Output,HC10_Out2
5,HC10,Output,HC10_Out3
6,HC20,Input,HC20_In1
7,HC20,Input,HC20_In2
8,HC20,Input,HC20_In3
9,HC20,Output,HC20_Out1



Module Maximum Volumes


,Module,Maximum Volume (L)
0,HC10,10.0
1,HC20,15.0
2,HC30,10.0
3,HC40,30.0



Module Initial Resources


,Module,Material,Quantity (L)
0,HC10,A,10.0
1,HC20,B,10.0
2,HC30,C,10.0


## Recipe and RTN Model

Write the recipe in the `RecipeIR` layer (parallel / choice / mix / dose / ...), then lift it into the solver-facing `RTNModel`. The solver consumes the `RTNModel` directly.

In [4]:
# ---- Step 1: Define an editable flat General Recipe ----
recipe_spec = {
    "id": "Module_Parallel_Choice_Example",
    "volume": 6.0,
    "procedure": [
        {"dose": {"ingredient": "A", "amount_L": 1.0}},
        {"dose": {"ingredient": "B", "amount_L": 2.0}},
        {"dose": {"ingredient": "C", "amount_L": 3.0}},
        {"mix": {"rpm": 150, "duration_s": 30}},
        {"usage": {"duration_s": 3600}},
        {"settling": {"duration_s": 300}},
        {"separation": {"order": ["C", "B", "A"]}},
    ],
}

general_ir = build_recipe_ir(recipe_spec)
display_recipe_graph(general_ir, "General Recipe Control Flow")

enriched_spec = auto_enrich_recipe_spec(recipe_spec)
recipe_ir = build_recipe_ir(enriched_spec)
display_recipe_graph(recipe_ir, "Enriched Recipe Control Flow")

generated_dir = PROJECT_ROOT / "generated" / "notebooks" / recipe_spec["id"]
generated_dir.mkdir(parents=True, exist_ok=True)
general_xml_path = save_general_recipe_xml_from_ir(
    general_ir,
    generated_dir / f"GeneralRecipe_{general_ir.id}.xml",
)

print(f"General Recipe : {len(general_ir.nodes)} nodes / {len(general_ir.edges)} edges")
print(f"Planning IR    : {len(recipe_ir.nodes)} nodes / {len(recipe_ir.edges)} edges")
print(f"Choice groups  : {recipe_ir.choice_groups()}")
print(f"General XML    : {general_xml_path}")

General Recipe : 9 nodes / 8 edges
Planning IR    : 14 nodes / 16 edges
Choice groups  : {'AND_001': ['b1', 'b2', 'b3'], 'XOR_008': ['fast', 'standard']}
General XML    : /Users/bowen/PycharmProjects/ModPlant-RTN/generated/notebooks/Module_Parallel_Choice_Example/GeneralRecipe_Module_Parallel_Choice_Example.xml


## Solve with the desktop App service

`RTNSettings`, Module conversion, RTN compilation, CP-SAT, energy/CO₂ costs, persistent Connect/Disconnect actions, and validation are executed by the shared ModPlant-RTN service. Master Recipe generation is an explicit export step.

In [5]:
# Edit these values exactly as in the desktop App Settings page.
app_settings = RTNSettings(
    solver_time_limit_s=180,
    num_workers=8,
    time_penalty_per_second=0.5,
    profit_per_litre=300.0,
    usage_cost_weight=1.0,
    energy_cost_weight=1.0,
    co2_penalty=1.0,
    electricity_price_eur_per_kwh=0.3,
    connect_duration_s=3.0,       # fallback only
    disconnect_duration_s=2.0,    # fallback only
    require_final_disconnect=False,
    relative_gap_limit=0.001,
    enable_auxiliary_transfers=True,
    allow_process_transfers=True,
    auxiliary_transfer_mode="lazy",
)

plant = plant_model_from_rtn_inputs(
    module_ops,
    module_interfaces,
    module_maximum_volume,
    module_resources,
    equipment_bindings=equipment_bindings,
)

def show_progress(stage, progress, detail):
    print(f"[{progress:6.1%}] {stage:10s} {detail}")

session = optimize_recipe_ir(
    recipe_ir,
    plant,
    recipe_path=general_xml_path,
    output_dir=generated_dir,
    settings=app_settings,
    progress_callback=show_progress,
)
planner_result = session.pipeline.planner_result
validation = session.pipeline.validation_result

[  2.0%] aas        Validating editable Module capabilities, ports and inventory
[ 22.0%] rtn        Compiling PlanningGraph and plant capabilities into the RTN model
[ 34.0%] solve      CP-SAT is selecting Module, routes, timing and nonlinear branches
[ 82.0%] validation Validating precedence, inventory, capacity, ports and branch semantics
[100.0%] complete   Optimization and validation completed; Master Recipe export is available on request


In [6]:
print("Status                    :", planner_result.status)
print("Selected branches         :", planner_result.selected_branches)
print("Makespan (s)              :", planner_result.makespan_s)
print("Weighted usage cost (€)   :", round(planner_result.total_weighted_usage_cost, 2))
print("Weighted energy cost (€)  :", round(planner_result.total_weighted_energy_cost, 2))
print("Weighted CO₂ cost (€)     :", round(planner_result.total_weighted_co2_cost, 2))
print("Total cost (€)            :", round(planner_result.total_cost, 2))
print("Revenue objective (€)     :", round(planner_result.objective_profit, 2))
print("Shared service elapsed (s):", round(session.elapsed_s, 3))
print("Connect / Disconnect      :", sum(op.operation_type == "connect" for op in planner_result.operations), "/", sum(op.operation_type == "disconnect" for op in planner_result.operations))

display_process_plan_graph(recipe_ir, planner_result, "CP-SAT Selected Process Plan (Gantt)")

# Dynamic App rows: new App result fields appear here without a Notebook column migration.
plan_df = plan_dataframe(session)
display_df("CP-SAT Selected Process Plan", plan_df)

Status                    : OPTIMAL
Selected branches         : {'XOR_008': 'fast'}
Makespan (s)              : 2283.0
Weighted usage cost (€)   : 42.0
Weighted energy cost (€)  : 6.88
Weighted CO₂ cost (€)     : 9.18
Total cost (€)            : 58.06
Revenue objective (€)     : 600.44
Shared service elapsed (s): 3.767
Connect / Disconnect      : 5 / 0



CP-SAT Selected Process Plan


,Step,Recipe Node,Branch Group,Branch,Operation Type,Transfer Kind,Operation,Module,Source Module,Target Module,...,Energy Consumption (kWh),Energy Cost,CO2 Emissions (kg),Weighted Usage Cost,Weighted Energy Cost,Weighted CO2 Cost,Total Cost,Material,OPC UA Endpoint,Namespace URI
0,1,AUX_006_CONNECT,,,connect,,"Connect(HC30_Out1 -> HC40_In1, 3.0s)",HC30->HC40,HC30,HC40,...,0.112500,0.03,0.045000,4.0,0.03,0.04,4.08,{},,
1,2,dose_001_CONNECT,,,connect,,"Connect(HC10_Out1 -> HC30_In1, 3.0s)",HC10->HC30,HC10,HC30,...,0.083333,0.02,0.033333,4.0,0.02,0.03,4.06,{},,
2,3,AUX_006,,,aux_transfer,pure_material,"Pure material Transfer: Draining(HC30), Fillin...",HC40,HC30,HC40,...,6.756944,2.03,2.702778,3.0,2.03,2.70,7.73,{'C': 7.0},opc.tcp://localhost:4840/HC40,urn:modplant:hc40
3,4,dose_002_CONNECT,,,connect,,"Connect(HC20_Out1 -> HC30_In2, 3.0s)",HC20->HC30,HC20,HC30,...,0.093750,0.03,0.037500,4.0,0.03,0.04,4.07,{},,
4,5,separation_002_CONNECT,,,connect,,"Connect(HC30_Out3 -> HC20_In1, 3.0s)",HC30->HC20,HC30,HC20,...,0.093750,0.03,0.037500,4.0,0.03,0.04,4.07,{},,
5,6,separation_003_CONNECT,,,connect,,"Connect(HC30_Out2 -> HC10_In1, 3.0s)",HC30->HC10,HC30,HC10,...,0.083333,0.02,0.033333,4.0,0.02,0.03,4.06,{},,
6,7,dose_001,AND_001,b1,dose,pure_material,"Dosing: Open Valve of HC10_Out1 only, Draining...",HC30,HC10,HC30,...,0.819444,0.25,0.327778,3.0,0.25,0.33,3.57,{'A': 1.0},opc.tcp://localhost:4840/HC30,urn:modplant:hc30
7,8,dose_002,AND_001,b2,dose,pure_material,"Dosing: Open Valve of HC20_Out1 only, Draining...",HC30,HC20,HC30,...,1.944444,0.58,0.777778,3.0,0.58,0.78,4.36,{'B': 2.0},opc.tcp://localhost:4840/HC30,urn:modplant:hc30
8,9,mix_001,,,mix,,"Stirring (HC30), 150rpm for 30.0s",HC30,HC30,HC30,...,2.500000,0.75,1.000000,3.0,0.75,1.00,4.75,{},opc.tcp://localhost:4840/HC30,urn:modplant:hc30
9,10,usage_001,XOR_008,fast,usage,,"Usage (HC30), 1800.0s: None",HC30,HC30,HC30,...,4.000000,1.20,1.600000,0.0,1.20,1.60,2.80,{},opc.tcp://localhost:4840/HC30,urn:modplant:hc30


## Inter-module connection configuration

The same solve that produces the Gantt also fixes the port-to-port wiring between modules. The connection graph below shows that configuration directly, the way the desktop App's connection page does: each arrow is a directed connection the plan establishes, and the connect operations on the Gantt are these edges placed in time.

In [7]:
display_connection_graph(plant, planner_result, "Solved Module Connections")

## Validation produced by the shared App pipeline

In [8]:
print("Valid                   :", validation.valid)
print("Errors                  :", len(validation.errors))
for error in validation.errors:
    print("  -", error)
print("Warnings                :", len(validation.warnings))
for warning in validation.warnings:
    print("  -", warning)

print("\nObjective recomputation:")
for key, value in validation.profit_check.items():
    print(f"  {key:24s}: {value}")
profit_match = abs(validation.profit_check["profit"] - planner_result.objective_profit) < 1e-3
print("\nPlanner objective       :", planner_result.objective_profit)
print("Validator objective     :", validation.profit_check["profit"])
print("Match                   :", profit_match)
print("Master Recipe generated :", session.pipeline.master_xml_path or "No — call export_master_recipe(...) explicitly")

Valid                   : True
Errors                  : 0
Warnings                : 0

Objective recomputation:
  base_profit             : 1800.0
  lambda_per_second       : -0.5
  total_duration_s        : 2305.0
  makespan_s              : 2283.0
  total_cost              : 58.062083333333334
  profit                  : 600.4379166666666

Planner objective       : 600.4379166666666
Validator objective     : 600.4379166666666
Match                   : True
Master Recipe generated : No — call export_master_recipe(...) explicitly
